# Demo (Optional) - Handoff Summary From a Real Diff
**Day 1 - Session 1, Topic 3**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/agent-orchestration-companion/blob/main/day1/demos-notebook/demo-handoff-summary.ipynb)

**Goal:** Turn a real Git diff into the written summary line of a handoff artifact, using a real LLM via API key (if provided), falling back to a small local model.

Everything structural in a handoff can be measured: the commit, the changed files, the test result. Only the reviewer-facing summary needs prose, and this notebook generates that from the actual diff.

**This is the optional demo.** Skip it when class time is short; the handoff structure is already covered by `demo-two-session-orchestration`.

> A small local model is weak. It is used for description only, never for the routing decision. If an API key is set via `.env` (e.g. `ANTHROPIC_API_KEY`), it uses the cloud model. Otherwise, it safely falls back.


## 1. Setup

Locate the course files and put `demo_support` on the import path.


In [1]:
# Setup: make the course files and demo_support importable.
# On Colab nothing is present yet, so clone the companion repo once.
# Locally this finds your existing checkout and clones nothing.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kpassoubady/agent-orchestration-companion.git"
MARKER = Path("lab-workspace-solution") / "router.py"


def find_repo_root():
    directory = Path.cwd()
    for _ in range(6):
        if (directory / MARKER).exists():
            return directory
        directory = directory.parent
    clone = Path.cwd() / "agent-orchestration-companion"
    if not (clone / MARKER).exists():
        print(f"Cloning {{REPO_URL}} ...")
        subprocess.run(
            ["git", "clone", "--depth", "1", "-q", REPO_URL, str(clone)],
            check=True,
            env=dict(os.environ, GIT_TERMINAL_PROMPT="0"),
        )
    return clone


ROOT = find_repo_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "day1" / "demos"))

# Colab has no global Git identity; the demos set a local one per sandbox repo.
print("Course root:", ROOT)
print("Git:", subprocess.run(["git", "--version"], capture_output=True, text=True).stdout.strip())

Course root: /Users/kangs/code/github/agent-orchestration-companion
Git: git version 2.50.1 (Apple Git-155)


## 2. Install the optional dependency

Only needed for the model path. Skip this cell to see the deterministic fallback instead; the notebook completes either way.


In [2]:
# Optional. The notebook runs without this cell by using the deterministic summary.
# On Colab this takes a minute; the model download below is about 1 GB.


## 3. Load the local model, or fall back cleanly

Every failure mode here degrades to the deterministic path instead of raising.


In [3]:
MODEL_NAME = "google/flan-t5-base"



def load_api_model():
    """Return a generate() callable using litellm if configured, else None."""
    try:
        from llm_client import get_completion, PROVIDER
        import os
        if not any(os.getenv(k) for k in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GEMINI_API_KEY", "AZURE_API_KEY", "AZURE_AD_TOKEN"]):
            return None, None
            
        def generate(prompt):
            return get_completion([{"role": "user", "content": prompt}], tier="mini", max_tokens=100)
        
        return generate, f"{PROVIDER} API (mini tier)"
    except Exception:
        return None, None

def load_local_model():
    """Return a generate() callable, or None when the model is unavailable."""
    try:
        import warnings

        warnings.filterwarnings("ignore")
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    except ImportError:
        print("  transformers is not installed; using the deterministic summary.")
        return None

    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    except Exception as error:
        print(f"  {MODEL_NAME} could not be loaded ({type(error).__name__});")
        print("  using the deterministic summary.")
        return None

    model.eval()

    def generate(prompt):
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
        output = model.generate(**inputs, max_new_tokens=48)  # greedy, deterministic
        return tokenizer.decode(output[0], skip_special_tokens=True).strip()

    return generate


from demo_support import assert_true, git, heading, run_tests, sandbox, show_evidence

heading("Local model availability")
GENERATE, SOURCE = load_api_model()
if not GENERATE:
    GENERATE = load_local_model()
    SOURCE = MODEL_NAME if GENERATE else "deterministic fallback"



Local model availability
------------------------


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 5855.97it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


## 4. Produce a real commit to summarize

Edit the renderer, run the focused test, and commit. The diff below is genuine.


In [4]:
from pathlib import Path

TARGET = Path("channels") / "email.py"
EDIT = ("your order shipped", "your order has shipped")
FOCUSED_TEST = "tests.test_email"

sandbox_context = sandbox()
WORK, BASE = sandbox_context.__enter__()

path = WORK / TARGET
path.write_text(path.read_text().replace(*EDIT))
PASSED, _ = run_tests(FOCUSED_TEST, WORK)
git("commit", "-q", "-am", "Refine shipment email wording", cwd=WORK)
COMMIT = git("rev-parse", "--short", "HEAD", cwd=WORK)
CHANGED = git("diff", "--name-only", f"{BASE}..{COMMIT}", cwd=WORK).splitlines()
DIFF_TEXT = git("diff", f"{BASE}..{COMMIT}", "--unified=0", cwd=WORK)

heading("Produce a real commit to summarize")
show_evidence("commit", COMMIT)
show_evidence("changed files", ", ".join(CHANGED))
show_evidence("focused test", "passed" if PASSED else "FAILED")

heading("The real diff handed to the summarizer")
for line in DIFF_TEXT.splitlines():
    if line.startswith(("+", "-")) and not line.startswith(("+++", "---")):
        print(f"  {line}")


Produce a real commit to summarize
----------------------------------
  commit                     825e020
  changed files              channels/email.py
  focused test               passed

The real diff handed to the summarizer
--------------------------------------
  -        "body": f"Hello {customer_name}, your order shipped via {carrier}. Track it at {tracking_url}",
  +        "body": f"Hello {customer_name}, your order has shipped via {carrier}. Track it at {tracking_url}",


## 5. Generate the summary and reject bad output

A small model often echoes its input. That must never reach a handoff, so echoed or too-short output is rejected in favour of the deterministic summary.


In [5]:
ADDED = [line[1:].strip() for line in DIFF_TEXT.splitlines()
         if line.startswith("+") and not line.startswith("+++")]
REMOVED = [line[1:].strip() for line in DIFF_TEXT.splitlines()
           if line.startswith("-") and not line.startswith("---")]


def deterministic_summary(files, added, removed):
    return (f"Changed {', '.join(files)}: {len(added)} line(s) added, "
            f"{len(removed)} removed, altering the rendered shipment wording.")


def looks_like_diff_echo(text, diff_text):
    markers = ("diff --git", "index ", "@@", "+++", "---", "b/", "100644")
    return any(marker in text for marker in markers) or text[:20] in diff_text


def model_summary(generate, added, removed):
    """Ask about the changed lines only; raw diff text invites metadata echo."""
    prompt = (
        "A developer edited a notification template.\n"
        f"Old text: {removed[0] if removed else '(nothing)'}\n"
        f"New text: {added[0] if added else '(nothing)'}\n"
        "Question: What kind of change is this? Answer in one sentence."
    )
    return generate(prompt)


heading("Generated summary field")
if GENERATE:
    raw = model_summary(GENERATE, ADDED, REMOVED)
    show_evidence("model output", raw or "(empty)")
    rejected = not raw or len(raw.split()) < 4 or looks_like_diff_echo(raw, DIFF_TEXT)
    if rejected:
        SUMMARY = deterministic_summary(CHANGED, ADDED, REMOVED)
        SOURCE_USED = "deterministic fallback (model output rejected)"
        show_evidence("rejected because", "echoed input or too terse")
        show_evidence("using instead", SUMMARY)
    else:
        # The model contributes prose; the measured facts stay authoritative.
        SUMMARY = f"{raw.rstrip('.')}. {deterministic_summary(CHANGED, ADDED, REMOVED)}"
        SOURCE_USED = f"{MODEL_NAME} prose + measured diff facts"
else:
    SUMMARY = deterministic_summary(CHANGED, ADDED, REMOVED)
    SOURCE_USED = "deterministic fallback"
    show_evidence("fallback output", SUMMARY)


Generated summary field
-----------------------
  model output               The notification template was edited.


## 6. Assemble the handoff artifact and verify it

Structural fields are measured; only the summary is generated. `summary_source` records which path ran.


In [6]:
import json

handoff = {
    "task_id": "email-renderer",
    "base_commit": BASE,
    "implementation_commit": COMMIT,
    "changed_files": sorted(CHANGED),
    "summary": SUMMARY,
    "summary_source": SOURCE_USED,
    "verification": {
        "command": f"python3 -m unittest {FOCUSED_TEST}",
        "status": "passed" if PASSED else "failed",
    },
    "unresolved_risks": [],
}
written = WORK / "handoffs" / "email.json"
written.write_text(json.dumps(handoff, indent=2) + "\n")

heading("Assembled handoff artifact")
print(json.dumps(handoff, indent=2))

heading("Evidence checks")
assert_true(DIFF_TEXT.strip() != "", "the diff came from a real commit")
assert_true(any(EDIT[1] in line for line in ADDED), "the diff contains the actual wording change")
assert_true(len(SUMMARY.split()) >= 4, "the summary field is reviewable prose")
assert_true(not looks_like_diff_echo(SUMMARY, DIFF_TEXT),
            "the summary is prose, not echoed diff metadata")
assert_true(json.loads(written.read_text())["implementation_commit"] == COMMIT,
            "the handoff on disk records the real commit")

sandbox_context.__exit__(None, None, None)
print("\nTakeaway: Measure the structural handoff fields from Git and tests,")
print("and generate only the prose summary, from the real diff.")


Assembled handoff artifact
--------------------------
{
  "task_id": "email-renderer",
  "base_commit": "d589480",
  "implementation_commit": "825e020",
  "changed_files": [
    "channels/email.py"
  ],
  "summary": "The notification template was edited. Changed channels/email.py: 1 line(s) added, 1 removed, altering the rendered shipment wording.",
  "summary_source": "google/flan-t5-base prose + measured diff facts",
  "verification": {
    "command": "python3 -m unittest tests.test_email",
    "status": "passed"
  },
  "unresolved_risks": []
}

Evidence checks
---------------
  [verified] the diff came from a real commit
  [verified] the diff contains the actual wording change
  [verified] the summary field is reviewable prose
  [verified] the summary is prose, not echoed diff metadata
  [verified] the handoff on disk records the real commit

Takeaway: Measure the structural handoff fields from Git and tests,
and generate only the prose summary, from the real diff.


### Expected output

- With the model cached: `summary source` is `google/flan-t5-base` and the
  model returns a short sentence such as "The notification template was
  edited.", which is then paired with the measured diff facts.
- Without `transformers` or the model: a clear one-line reason, then
  `summary source` is `deterministic fallback`. The notebook still completes.
- The real diff shows one removed and one added `body` line.
- The handoff JSON carries the real commit, the changed file, the test status,
  and an honest `summary_source`.
- Five `[verified]` lines, then the takeaway.
